In [1]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image

# 1. Tải model VGG16 đã train sẵn trên ImageNet
vgg16 = models.vgg16(weights="DEFAULT")

# 2. Giữ lại phần trích xuất đặc trưng (Features) và một phần Classifier (đến lớp fc2)
# Chúng ta loại bỏ lớp phân loại cuối cùng (lớp thứ 7 trong classifier)
model_vgg16 = vgg16
model_vgg16.classifier = torch.nn.Sequential(*(list(vgg16.classifier.children())[:-1]))

model_vgg16.eval() # Chuyển sang chế độ dự đoán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_vgg16.to(device)

def get_vgg16_features(img_path):
    # Chuẩn hóa ảnh theo chuẩn ImageNet (giống ResNet)
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    try:
        img = Image.open(img_path).convert('RGB')
        img_t = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            features = model_vgg16(img_t)

        return features.squeeze().cpu().numpy() # Kết quả: 4096 chiều
    except Exception as e:
        print(f"Lỗi khi xử lý ảnh {img_path}: {e}")
        return None

# Ví dụ sử dụng:
# feat = get_vgg16_features("path_to_your_product_image.jpg")
# print(feat.shape) # (4096,)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 186MB/s]


In [2]:
!wget https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png

--2026-03-31 09:13:44--  https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252444 (247K) [image/png]
Saving to: ‘CLIP.png’

CLIP.png            100%[===================>] 246.53K  --.-KB/s    in 0.01s   

2026-03-31 09:13:44 (16.1 MB/s) - ‘CLIP.png’ saved [252444/252444]



In [3]:
feat = get_vgg16_features("CLIP.png")
# print(feat.shape) # (4096,)

In [4]:
feat.shape

(4096,)

In [14]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/code/preprocessing/data"

In [16]:
!ls "{PATH}/2023/step4_image_feature_extract"

df_filtered_image.parquet  filtered_images.tar.gz


In [17]:
!cp "{PATH}/2023/step4_image_feature_extract/filtered_images.tar.gz" "filtered_images.tar.gz"

In [18]:
# !unzip downloaded_images.zip

In [21]:
!mkdir -p filtered_images
!tar -xzf filtered_images.tar.gz -C filtered_images

### Tạo file `image_feature.b`

In [22]:
import os
import numpy as np
import array
from tqdm.notebook import tqdm # Import tqdm for progress bar

In [23]:
def read_image_features(path, feature_size):
    if not os.path.exists(path): return
    with open(path, 'rb') as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes: break
            try:
                asin = asin_bytes.decode('utf-8').strip()
                a = array.array('f')
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [24]:
def extract_and_write_features(image_directory, output_file_path, existing_asins_set, feature_extractor_fn, expected_size):
    image_files = [f for f in os.listdir(image_directory) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    total_images = len(image_files)
    processed_count = 0
    added_count = 0
    already_existing_count = 0
    invalid_asin_length_count = 0
    size_mismatch_count = 0 # Biến mới: Đếm lỗi sai kích thước vector

    print(f"Tìm thấy {total_images} hình ảnh. Đang trích xuất với size kỳ vọng: {expected_size}...")

    with open(output_file_path, 'ab') as f_out:
        for image_file in tqdm(image_files, desc="Trích xuất đặc trưng", unit="ảnh"):
            asin = os.path.splitext(image_file)[0].strip()
            img_path = os.path.join(image_directory, image_file)

            # 1. Check độ dài ASIN
            if len(asin) != 10:
                invalid_asin_length_count += 1
                processed_count += 1
                continue

            # 2. Check trùng
            if asin in existing_asins_set:
                already_existing_count += 1
                processed_count += 1
                continue

            # 3. Trích xuất
            features = feature_extractor_fn(img_path)

            if features is not None:
                try:
                    # Chuyển về numpy array 1D
                    features = np.array(features).flatten().astype(np.float32)

                    # KIỂM TRA ĐỘ DÀI TRƯỚC KHI GHI (Cực kỳ quan trọng)
                    if len(features) != expected_size:
                        size_mismatch_count += 1
                        processed_count += 1
                        continue

                    # Ghi ASIN (10 bytes)
                    f_out.write(asin[:10].encode('utf-8'))

                    # Ghi đúng số lượng float của model đó ra bytes
                    f_out.write(features.tobytes())

                    added_count += 1
                    existing_asins_set.add(asin)
                    f_out.flush() # Ghi ngay xuống đĩa
                except Exception as e:
                    print(f"Lỗi khi ghi ASIN '{asin}': {e}")
            else:
                print(f"Không trích xuất được ASIN '{asin}'.")

            processed_count += 1

    print("\n--- BÁO CÁO CHI TIẾT ---")
    print(f"Tổng số hình ảnh đã xử lý: {processed_count}")
    print(f"Số đặc trưng đã thêm mới: {added_count}")
    print(f"Số ASIN đã có sẵn (bỏ qua): {already_existing_count}")
    print(f"Số ASIN sai độ dài (không phải 10): {invalid_asin_length_count}")
    print(f"Số ảnh sai kích thước vector ({expected_size}): {size_mismatch_count}")

In [25]:
def main_feature_extraction(output_binary_file_name, image_directory, model_fn, feature_size):
    # 1. Load các ASIN cũ (Phải truyền đúng size của model đã dùng cho file đó)
    existing_asins = set()
    if os.path.exists(output_binary_file_name):
        print(f"Đang kiểm tra dữ liệu cũ (Size: {feature_size})...")
        for asin, _ in read_image_features(output_binary_file_name, feature_size):
            existing_asins.add(asin)

    # 2. Chạy trích xuất mới
    extract_and_write_features(image_directory, output_binary_file_name, existing_asins, model_fn, feature_size)

# --- VÍ DỤ SỬ DỤNG ---

# Nếu dùng VGG16:
# main_feature_extraction('vgg_features.b', 'download_image', get_vgg16_features, 4096)

# Nếu dùng ResNet50:
# main_feature_extraction('resnet_features.b', 'download_image', get_resnet_features, 2048)

In [ ]:
# --- Script entry point ---
if __name__ == "__main__":
    output_binary_file = 'image_feature.b'
    image_dir = 'filtered_images'
    main_feature_extraction(output_binary_file, image_dir, get_vgg16_features, 4096)

Tìm thấy 35979 hình ảnh. Đang trích xuất với size kỳ vọng: 4096...


Trích xuất đặc trưng:   0%|          | 0/35979 [00:00<?, ?ảnh/s]

In [ ]:
!cp "/content/image_feature.b" "{PATH}/2023/step4_image_feature_extract/image_feature.vgg16.b"

### Kiểm tra file vừa tạo bằng hàm `readImageFeatures` của bạn

In [ ]:
import array

def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin_bytes = f.read(10)
    if not asin_bytes: break # Kiểm tra nếu không còn bytes để đọc
    asin = asin_bytes.decode('utf-8')
    a = array.array('f')
    try:
        a.fromfile(f, 4096)
        yield asin, a.tolist()
    except EOFError: # Bắt lỗi khi không đủ 4096 float để đọc
        print(f"Lỗi: Không đủ dữ liệu float cho ASIN '{asin}'. Có thể file bị hỏng.")
        break


# Đọc và kiểm tra file 'image_feature.b' vừa tạo
print(f"\nKiểm tra nội dung file '{output_binary_file}' bằng hàm của bạn:")
count = 0
for asin, features in readImageFeatures(output_binary_file):
    print(f"ASIN: {asin}, Kiểu dữ liệu: {type(features)}, Kích thước: {len(features)}")
    if count < 2: # Chỉ in chi tiết 3 mục đầu để không quá dài
        print(f"  Một vài giá trị đầu tiên: {features[:5]}")
    count += 1
    if count >= 3: # Giới hạn số lượng mục in ra
        print("... và nhiều mục khác.")
        break

print(f"Tổng số mục đã đọc: {count}")



Kiểm tra nội dung file 'image_feature.b' bằng hàm của bạn:
ASIN: B004LE8ZYO, Kiểu dữ liệu: <class 'list'>, Kích thước: 4096
  Một vài giá trị đầu tiên: [0.0, 0.0, 0.0, 0.0, 0.0]
ASIN: B0080HPWMG, Kiểu dữ liệu: <class 'list'>, Kích thước: 4096
  Một vài giá trị đầu tiên: [0.0, 0.0, 0.0, 0.0, 0.0]
ASIN: B008RJ9BH4, Kiểu dữ liệu: <class 'list'>, Kích thước: 4096
... và nhiều mục khác.
Tổng số mục đã đọc: 3
